# INEGI — archivos anuales y catálogos reales

Se trabajan los archivos `atus_anual_*.csv` encontrados en la carpeta real.  
Para **INEGI** el proyecto utiliza 2018 en adelante. Para **Kaggle**, la fuente se conserva como histórica/nacional y se prepara el subconjunto útil para CDMX.

Los catálogos se leen por separado y no se concatenan con los hechos, porque son tablas de referencia.


## Principios de limpieza y trazabilidad — INEGI

En esta fuente se aplicarán las siguientes reglas:

- **Conservar el valor original.** Las columnas originales no se sobrescriben.
- **Crear columnas limpias separadas.** Una transformación produce una nueva columna identificable.
- **No asumir equivalencias.** Que dos valores, campos o registros se repitan no significa que representen exactamente lo mismo.
- **No eliminar folios por repetición.** Un folio repetido se marca para revisión, pero no se descarta automáticamente.
- **No inventar faltantes.** Un dato vacío continúa como faltante salvo que exista una regla documentada de la fuente.
- **Separar únicamente copias exactas cuando corresponda.** Se conserva la primera aparición y las copias exactas adicionales se envían a una tabla de excluidos.
- **Documentar qué se conserva y qué se separa.**
- **Registrar cada decisión en una bitácora.**


## Delimitación temporal del proyecto: 2018 en adelante

Para mantener coherencia entre las fuentes, la **base analítica del proyecto se delimita a 2018 en adelante**.

Los datos anteriores a 2018:
- no se consideran datos incorrectos;
- no se eliminan definitivamente;
- no se mezclan con la base analítica 2018+;
- se conservan en archivos históricos separados;
- se documentan como **fuera del alcance temporal del análisis**.

La limpieza, los indicadores de calidad, la revisión de faltantes, formatos y duplicados se aplican a la base correspondiente al periodo **2018 en adelante**.


## 0. Cargar datos

En esta sección se carga `CARPETA_DATOS.zip` y se prepara el entorno de trabajo. El archivo comprimido se utiliza porque las bases originales son pesadas. Aquí todavía no se modifica ningún registro.


In [ ]:
from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile, unicodedata, re

subidos = files.upload()
nombre_zip = next(iter(subidos))

with zipfile.ZipFile(nombre_zip) as z:
    z.extractall("/content")

ROOT = Path("/content/CARPETA_DATOS")

def leer_csv(ruta):
    for enc in ["utf-8-sig","utf-8","latin-1","cp1252"]:
        for sep in [",",";"]:
            try:
                d = pd.read_csv(ruta, encoding=enc, sep=sep, low_memory=False)
                if len(d.columns) > 1:
                    return d
            except Exception:
                pass
    raise ValueError(f"No se pudo leer: {ruta}")

def texto_limpio(s):
    return (s.astype("string").str.strip().str.replace(r"\s+"," ",regex=True)
            .replace({"":pd.NA,"nan":pd.NA,"None":pd.NA,"NULL":pd.NA,"null":pd.NA}))

def normalizar(v):
    if pd.isna(v): return pd.NA
    v=" ".join(str(v).upper().strip().split())
    return "".join(c for c in unicodedata.normalize("NFD",v)
                   if unicodedata.category(c)!="Mn")

ALCALDIAS = {
"ALVARO OBREGON":"Álvaro Obregón","AZCAPOTZALCO":"Azcapotzalco",
"BENITO JUAREZ":"Benito Juárez","COYOACAN":"Coyoacán",
"CUAJIMALPA DE MORELOS":"Cuajimalpa de Morelos","CUAUHTEMOC":"Cuauhtémoc",
"GUSTAVO A MADERO":"Gustavo A. Madero","IZTACALCO":"Iztacalco",
"IZTAPALAPA":"Iztapalapa","LA MAGDALENA CONTRERAS":"La Magdalena Contreras",
"MIGUEL HIDALGO":"Miguel Hidalgo","MILPA ALTA":"Milpa Alta",
"TLAHUAC":"Tláhuac","TLALPAN":"Tlalpan",
"VENUSTIANO CARRANZA":"Venustiano Carranza","XOCHIMILCO":"Xochimilco"}


## 1. Localizar archivos anuales

En esta sección se identifican los archivos por año y se aplica la delimitación temporal. Los años **2018 en adelante** continúan hacia la limpieza. Los años anteriores a 2018 no se consideran errores ni se eliminan: se conservan como información histórica separada.


In [ ]:
BASE=ROOT/"INEGI"/"conjunto_de_datos_atus_anual_csv"/"conjunto_de_datos"
rutas=sorted(BASE.glob("atus_anual_*.csv"))
seleccion=[]
for p in rutas:
    m=re.search(r"(\\d{4})",p.stem)
    if m and int(m.group(1)) >= 2018:
        seleccion.append(p)
print("Archivos seleccionados:",len(seleccion))
for p in seleccion: print(p.name)


## 2. Leer y unir años conservando archivo de origen

En esta sección se leen los CSV seleccionados para el periodo analítico y se conserva el nombre del archivo de origen y la fila de procedencia. Esto permite rastrear cada registro hasta su archivo original.


## Separar archivos INEGI anteriores a 2018

En esta sección se identifican los archivos por año y se aplica la delimitación temporal. Los años **2018 en adelante** continúan hacia la limpieza. Los años anteriores a 2018 no se consideran errores ni se eliminan: se conservan como información histórica separada.


In [ ]:
# Los archivos 1997–2017 se conservan como histórico y NO entran a la limpieza analítica.
todas_rutas=sorted(BASE.glob("atus_anual_*.csv"))
rutas_historicas=[]
rutas_analiticas=[]

for p in todas_rutas:
    m=re.search(r"(\d{4})",p.stem)
    if m:
        anio=int(m.group(1))
        if anio < 2018:
            rutas_historicas.append(p)
        else:
            rutas_analiticas.append(p)

print("Archivos históricos (<2018):",len(rutas_historicas))
print("Archivos analíticos (2018+):",len(rutas_analiticas))


## Exportar INEGI histórico en un CSV aparte

Aquí se guardan aparte los registros anteriores a 2018. Esta separación permite conservar la información original sin mezclarla con el periodo analítico del proyecto. Estos datos quedan disponibles como evidencia histórica.


In [ ]:
HISTORICO=Path("/content/INEGI_HISTORICO_ANTES_2018")
HISTORICO.mkdir(exist_ok=True)

partes_historicas=[]
for p in rutas_historicas:
    d=leer_csv(p)
    d.insert(0,"_archivo_origen",p.name)
    partes_historicas.append(d)

if partes_historicas:
    inegi_historico=pd.concat(partes_historicas,ignore_index=True,sort=False)
    inegi_historico.to_csv(HISTORICO/"INEGI_1997_2017_historico.csv",
                           index=False,encoding="utf-8-sig")
    print("Filas históricas conservadas:",len(inegi_historico))

# La variable seleccion se redefine explícitamente para la limpieza 2018+.
seleccion=rutas_analiticas


In [ ]:
partes=[]
for p in seleccion:
    d=leer_csv(p)
    d.insert(0,"_archivo_origen",p.name)
    d.insert(1,"_fila_archivo",np.arange(2,len(d)+2))
    partes.append(d)
datos=pd.concat(partes,ignore_index=True,sort=False)
print(datos.shape)


## 3. Leer catálogos por separado

Los catálogos se cargan como tablas de referencia independientes. No se concatenan con los hechos porque su función es ayudar a interpretar códigos de entidad, municipio, periodo u otras categorías sin inventar significados.


In [ ]:
CAT=ROOT/"INEGI"/"conjunto_de_datos_atus_anual_csv"/"catalogos"
catalogos={}
if CAT.exists():
    for p in sorted(CAT.glob("*.csv")):
        catalogos[p.stem]=leer_csv(p)
        print(p.name,catalogos[p.stem].shape)


## 4. Limpiar año

Se crea una versión numérica limpia del año sin sobrescribir el valor original. Los valores que no puedan interpretarse permanecen identificables para su revisión.


In [ ]:
datos["ANIO_limpio"]=pd.to_numeric(datos["ANIO"],errors="coerce").astype("Int64")


## 5. Limpiar mes

Se transforma el mes a formato numérico en una columna nueva y se marcan los valores fuera del rango esperado. No se inventa un mes cuando el dato está vacío o no es válido.


In [ ]:
datos["MES_limpio"]=pd.to_numeric(datos["MES"],errors="coerce").astype("Int64")
datos["_mes_fuera_rango"]=~datos["MES_limpio"].between(1,12) & datos["MES_limpio"].notna()


## 6. Limpiar día

Se prepara el día en una columna limpia y se identifican valores fuera del rango esperado. El valor original se conserva para mantener trazabilidad.


In [ ]:
datos["ID_DIA_limpio"]=pd.to_numeric(datos["ID_DIA"],errors="coerce").astype("Int64")
datos["_dia_fuera_rango"]=~datos["ID_DIA_limpio"].between(1,31) & datos["ID_DIA_limpio"].notna()


## 7. Limpiar hora

Se convierte la hora a formato numérico sin sustituir el campo original. Las horas que no puedan interpretarse o estén fuera de rango se marcan, pero la fila no se elimina automáticamente.


In [ ]:
datos["ID_HORA_limpio"]=pd.to_numeric(datos["ID_HORA"],errors="coerce").astype("Int64")
datos["_hora_fuera_rango"]=~datos["ID_HORA_limpio"].between(0,23) & datos["ID_HORA_limpio"].notna()


## 8. Crear franja horaria

A partir de una hora válida se crea una variable derivada de franja horaria. Esta columna ayudará a comparar la frecuencia de hechos por horario sin modificar la hora original.


In [ ]:
hora_valida=datos["ID_HORA_limpio"].where(datos["ID_HORA_limpio"].between(0,23))
datos["franja_horaria"]=pd.cut(hora_valida,[-1,5,11,17,23],
 labels=["Madrugada","Mañana","Tarde","Noche"])


## 9. Limpiar entidad

Se prepara el código de entidad en una columna limpia. El código original permanece intacto y posteriormente puede contrastarse con los catálogos oficiales de la propia fuente.


In [ ]:
datos["CVE_ENT_limpio"]=pd.to_numeric(datos["CVE_ENT"],errors="coerce").astype("Int64")


## 10. Limpiar municipio

Se prepara el código de municipio/alcaldía en una columna nueva. No se asume que un código significa una alcaldía específica sin apoyarse en el catálogo correspondiente.


In [ ]:
datos["CVE_MUN_limpio"]=pd.to_numeric(datos["CVE_MUN"],errors="coerce").astype("Int64")


## 11. Calcular personas fallecidas

Se convierten a formato numérico los componentes relacionados con personas fallecidas y se calcula una medida derivada únicamente cuando existen datos. Si todos los componentes están vacíos, el resultado permanece faltante en lugar de convertirse artificialmente en cero.


In [ ]:
cols_muertos=["CONDMUERTO","PASAMUERTO","PEATMUERTO","CICLMUERTO","OTROMUERTO","NEMUERTO"]
for c in cols_muertos:
    if c in datos: datos[c+"_limpio"]=pd.to_numeric(datos[c],errors="coerce")
mu=[c+"_limpio" for c in cols_muertos if c+"_limpio" in datos]
datos["personas_fallecidas_calculadas"]=datos[mu].sum(axis=1,min_count=1)


## 12. Calcular personas heridas

Se preparan los componentes relacionados con personas heridas/lesionadas y se obtiene una medida derivada respetando los faltantes. No se interpreta un dato ausente como cero.


In [ ]:
cols_heridos=["CONDHERIDO","PASAHERIDO","PEATHERIDO","CICLHERIDO","OTROHERIDO","NEHERIDO"]
for c in cols_heridos:
    if c in datos: datos[c+"_limpio"]=pd.to_numeric(datos[c],errors="coerce")
he=[c+"_limpio" for c in cols_heridos if c+"_limpio" in datos]
datos["personas_heridas_calculadas"]=datos[he].sum(axis=1,min_count=1)


## 13. Marcar duplicados exactos

Se identifican copias exactas de filas. Una repetición parcial o de un identificador no se considera automáticamente un duplicado. Cuando existe una copia exacta adicional, se conserva una primera aparición y la copia puede separarse para evitar doble conteo.


In [ ]:
originales=[c for c in datos.columns if not c.startswith("_") and not c.endswith("_limpio")
           and c not in ["franja_horaria","personas_fallecidas_calculadas","personas_heridas_calculadas"]]
datos["_duplicado_exacto"]=datos.duplicated(originales,keep="first")


## 14. Conservar registros 2018+ sin duplicado exacto

Se identifican copias exactas de filas. Una repetición parcial o de un identificador no se considera automáticamente un duplicado. Cuando existe una copia exacta adicional, se conserva una primera aparición y la copia puede separarse para evitar doble conteo.


In [ ]:
datos_limpios=datos.loc[~datos["_duplicado_exacto"]].copy()
print(datos_limpios.shape)


## 15. Exportar resultados como CSV individuales

Se guardan los resultados y evidencias en CSV independientes. De esta manera puedes utilizar la base limpia y, por separado, conservar los perfiles, conciliaciones y bitácoras.


In [ ]:
SALIDA=Path("/content/INEGI_RESULTADOS_CSV"); SALIDA.mkdir(exist_ok=True)
datos_limpios.to_csv(SALIDA/"INEGI_base_preparada.csv",index=False,encoding="utf-8-sig")
for origen,g in datos_limpios.groupby("_archivo_origen",dropna=False):
    g.to_csv(SALIDA/(Path(str(origen)).stem+"_limpio.csv"),index=False,encoding="utf-8-sig")
for p in sorted(SALIDA.glob("*.csv")): print(p.name)


## 16. Descargar un CSV individual

En esta sección se carga `CARPETA_DATOS.zip` y se prepara el entorno de trabajo. El archivo comprimido se utiliza porque las bases originales son pesadas. Aquí todavía no se modifica ningún registro.


In [ ]:
archivos_csv=sorted(SALIDA.glob("*.csv"))
for i,p in enumerate(archivos_csv,1): print(i,p.name)
n=int(input("Número: "))
if 1<=n<=len(archivos_csv): files.download(str(archivos_csv[n-1]))


## Auditoría 1 — Vacíos por columna

Se contabilizan los datos faltantes por columna y su porcentaje. Los vacíos se conservan como faltantes: no se sustituyen por cero, texto u otros valores sin evidencia.

Este bloque cuenta los valores faltantes sin modificarlos. Sirve para documentar qué campos tienen ausencia de información.


In [ ]:
perfil_vacios = []
objetos = {}

# Detectar automáticamente las tablas principales disponibles en este notebook.
for nombre in ["tablas_ssc","c5","datos","datos_limpios","cdmx"]:
    if nombre in globals():
        objetos[nombre] = globals()[nombre]

if "tablas_ssc" in objetos:
    for tabla, d in tablas_ssc.items():
        for col in d.columns:
            perfil_vacios.append({
                "tabla": tabla,
                "columna": col,
                "filas": len(d),
                "vacios": int(d[col].isna().sum()),
                "porcentaje_vacio": round(d[col].isna().mean()*100,2)
            })
else:
    # Elegir la tabla más completa disponible.
    d = c5 if "c5" in globals() else (datos if "datos" in globals() else (cdmx if "cdmx" in globals() else datos_limpios))
    for col in d.columns:
        perfil_vacios.append({
            "tabla": "base",
            "columna": col,
            "filas": len(d),
            "vacios": int(d[col].isna().sum()),
            "porcentaje_vacio": round(d[col].isna().mean()*100,2)
        })

perfil_vacios = pd.DataFrame(perfil_vacios)
display(perfil_vacios.sort_values(["tabla","porcentaje_vacio"],ascending=[True,False]))


## Auditoría 2 — Formatos diferentes o no interpretables

Se resumen los registros cuyo formato no pudo interpretarse o quedó fuera de un rango válido. El objetivo es documentar el problema sin ocultar ni borrar el valor original.

Las columnas de control creadas durante la limpieza (`_...invalida`, `_...fuera_rango`, etc.) permiten contar problemas de formato sin borrar el valor original.


In [ ]:
problemas_formato = []
bases_revision = tablas_ssc if "tablas_ssc" in globals() else {"base": (c5 if "c5" in globals() else datos)}

for tabla,d in bases_revision.items():
    controles = [c for c in d.columns if c.startswith("_") and any(
        palabra in c for palabra in ["invalida","invalido","fuera_rango","negativo","no_reconocida"]
    )]
    for c in controles:
        problemas_formato.append({
            "tabla": tabla,
            "control": c,
            "registros_afectados": int(d[c].fillna(False).astype(bool).sum())
        })

problemas_formato = pd.DataFrame(problemas_formato)
display(problemas_formato)


## Auditoría 3 — Duplicados exactos y valores repetidos

Se identifican copias exactas de filas. Una repetición parcial o de un identificador no se considera automáticamente un duplicado. Cuando existe una copia exacta adicional, se conserva una primera aparición y la copia puede separarse para evitar doble conteo.

Se distinguen dos situaciones: una copia exacta de toda la fila y un identificador/folio que aparece varias veces. Solo la primera situación puede separarse automáticamente.


In [ ]:
resumen_repetidos = []
bases_revision = tablas_ssc if "tablas_ssc" in globals() else {"base": (c5 if "c5" in globals() else datos)}

for tabla,d in bases_revision.items():
    duplicados = int(d["_duplicado_exacto"].sum()) if "_duplicado_exacto" in d else int(d.duplicated().sum())
    fila = {
        "tabla": tabla,
        "filas_totales": len(d),
        "copias_exactas_adicionales": duplicados,
        "filas_conservables_si_solo_separan_exactos": len(d)-duplicados
    }
    if "folio" in d.columns:
        fila["folios_distintos_repetidos"] = int(
            d.loc[d["folio"].notna() & d.duplicated("folio",keep=False),"folio"].nunique()
        )
        fila["filas_con_folio_repetido"] = int(
            (d["folio"].notna() & d.duplicated("folio",keep=False)).sum()
        )
    resumen_repetidos.append(fila)

resumen_repetidos = pd.DataFrame(resumen_repetidos)
display(resumen_repetidos)


## Auditoría 4 — Qué datos se conservan y cuáles se separan
La conciliación debe cumplir: **filas originales = filas conservadas + copias exactas adicionales separadas**. Los faltantes, formatos problemáticos y folios repetidos no se eliminan automáticamente.


In [ ]:
conciliacion_auditable = []
bases_revision = tablas_ssc if "tablas_ssc" in globals() else {"base": (c5 if "c5" in globals() else datos)}

for tabla,d in bases_revision.items():
    mascara_dup = d["_duplicado_exacto"] if "_duplicado_exacto" in d else d.duplicated(keep="first")
    conciliacion_auditable.append({
        "tabla": tabla,
        "filas_originales": len(d),
        "filas_conservadas": int((~mascara_dup).sum()),
        "filas_separadas": int(mascara_dup.sum()),
        "motivo_separacion": "Copia exacta adicional",
        "faltantes_eliminados": 0,
        "folios_repetidos_eliminados": 0
    })

conciliacion_auditable = pd.DataFrame(conciliacion_auditable)
conciliacion_auditable["comprobacion"] = (
    conciliacion_auditable["filas_originales"] ==
    conciliacion_auditable["filas_conservadas"] + conciliacion_auditable["filas_separadas"]
)
display(conciliacion_auditable)
assert conciliacion_auditable["comprobacion"].all()


## Bitácora de decisiones

La bitácora registra las decisiones tomadas durante la limpieza: qué se hizo, qué información se conservó, qué se separó y la justificación. Esto hace reproducible y defendible el proceso.

Esta bitácora resume las decisiones metodológicas. Los conteos de registros afectados se obtienen de las auditorías anteriores.


In [ ]:
bitacora_decisiones = pd.DataFrame([
    {
        "fuente":"INEGI",
        "decision":"Conservar valores originales",
        "accion":"No sobrescribir columnas crudas",
        "datos_que_se_conservan":"Todos los valores originales",
        "datos_que_se_separan":"Ninguno por esta regla",
        "justificacion":"Mantener trazabilidad y permitir comparar antes y después."
    },
    {
        "fuente":"INEGI",
        "decision":"Crear columnas limpias",
        "accion":"Agregar columnas derivadas sin reemplazar las originales",
        "datos_que_se_conservan":"Original + versión limpia",
        "datos_que_se_separan":"Ninguno",
        "justificacion":"Evitar pérdida de información."
    },
    {
        "fuente":"INEGI",
        "decision":"Mantener faltantes",
        "accion":"No imputar valores sin evidencia",
        "datos_que_se_conservan":"Filas con datos faltantes",
        "datos_que_se_separan":"Ninguno solo por estar vacío",
        "justificacion":"Vacío no equivale a cero ni a una categoría conocida."
    },
    {
        "fuente":"INEGI",
        "decision":"Mantener identificadores repetidos",
        "accion":"Marcar para revisión sin eliminar",
        "datos_que_se_conservan":"Registros con folio/identificador repetido",
        "datos_que_se_separan":"Ninguno solo por repetición",
        "justificacion":"La repetición del identificador no demuestra que las filas sean copias."
    },
    {
        "fuente":"INEGI",
        "decision":"Separar duplicados exactos",
        "accion":"Conservar primera aparición y separar copias exactas adicionales",
        "datos_que_se_conservan":"Primera copia y todas las filas no idénticas",
        "datos_que_se_separan":"Copias exactas adicionales",
        "justificacion":"Evitar doble conteo sin eliminar registros distintos."
    }
])
display(bitacora_decisiones)


## Exportar evidencias de auditoría como CSV

Se guardan los resultados y evidencias en CSV independientes. De esta manera puedes utilizar la base limpia y, por separado, conservar los perfiles, conciliaciones y bitácoras.

Estos archivos son pequeños y pueden descargarse individualmente junto con las bases limpias.


In [ ]:
# Usa la carpeta SALIDA que ya fue creada por el notebook.
perfil_vacios.to_csv(SALIDA/"perfil_vacios.csv",index=False,encoding="utf-8-sig")
problemas_formato.to_csv(SALIDA/"problemas_formato.csv",index=False,encoding="utf-8-sig")
resumen_repetidos.to_csv(SALIDA/"resumen_repetidos.csv",index=False,encoding="utf-8-sig")
conciliacion_auditable.to_csv(SALIDA/"conciliacion_auditable.csv",index=False,encoding="utf-8-sig")
bitacora_decisiones.to_csv(SALIDA/"bitacora_decisiones.csv",index=False,encoding="utf-8-sig")

print("Evidencias de auditoría generadas.")


# Resumen para documentar la limpieza

La limpieza se diseñó para conservar la trazabilidad de la información. Los valores originales permanecen disponibles y las transformaciones se almacenan en columnas nuevas. Los valores faltantes no se sustituyen por ceros, categorías ni otros valores inventados. Los formatos no interpretables se marcan mediante columnas de control y el registro se conserva para que el problema pueda revisarse.

Los valores repetidos se analizan según su significado. Un folio o identificador repetido no se considera automáticamente un duplicado, porque puede representar registros distintos relacionados con un mismo hecho. Por esta razón, los folios repetidos se contabilizan y marcan, pero no se eliminan. La separación automática se limita a copias exactas adicionales de una fila, conservando siempre una primera aparición.

Para demostrar qué ocurrió durante la preparación se generan tablas de auditoría con el número de vacíos por columna, problemas de formato, duplicados exactos, identificadores repetidos y una conciliación entre filas originales, conservadas y separadas. Finalmente, cada decisión metodológica se registra en una bitácora y todos estos productos pueden descargarse individualmente en formato CSV.


## Descargar individualmente los archivos históricos

En esta sección se carga `CARPETA_DATOS.zip` y se prepara el entorno de trabajo. El archivo comprimido se utiliza porque las bases originales son pesadas. Aquí todavía no se modifica ningún registro.


In [ ]:
CARPETA_HISTORICA=Path("/content/INEGI_HISTORICO_ANTES_2018")
archivos_historicos=sorted(CARPETA_HISTORICA.glob("*.csv"))

if not archivos_historicos:
    print("No se generaron archivos históricos para esta fuente.")
else:
    for i,p in enumerate(archivos_historicos,1):
        print(i,p.name,f"(0.02 MB)")

    n=int(input("Número del histórico a descargar: "))
    if 1 <= n <= len(archivos_historicos):
        files.download(str(archivos_historicos[n-1]))


## Descripción final de la delimitación temporal

Para **INEGI**, decidimos trabajar la limpieza analítica con los datos correspondientes a **2018 en adelante**, porque este es el periodo delimitado para el proyecto. Los registros anteriores a 2018 se conservaron en un archivo histórico independiente y no fueron tratados como errores ni eliminados definitivamente.

La separación temporal se realiza antes de aplicar las transformaciones principales. De esta forma, los indicadores de faltantes, formatos diferentes, duplicados y demás decisiones de limpieza describen el periodo que realmente se utilizará en el análisis. Al mismo tiempo, la información histórica permanece disponible para consulta y trazabilidad.
